# Import libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import numpy as np
import ucimlrepo as uciml

In [ ]:
%matplotlib inline

# Create Model Class that inherits nn.Module

In [ ]:
class Model(nn.Module):
    # Input Layer(4 neurons)
    # Hidden Layer(n neurons)
    # Output Layer(3 neurons)
    
    # define structure of neural network(number of neurons in each layer)
    def __init__(self, in_features, h1, h2, out_features):
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1) # connecting input layer to hidden layer 1
        self.fc2 = nn.Linear(h1, h2) # connecting hidden layer 1 to hidden layer 2
        self.out = nn.Linear(h2, out_features) # connecting hidden layer 2 to output layer
    
    # define forward function (how data flows through the network)
    def forward(self, x):
        x = F.relu(self.fc1(x)) # applying ReLU activation function to the output of hidden layer 1
        x = F.relu(self.fc2(x)) # applying ReLU activation function to the output of hidden layer 2
        x = self.out(x) # output layer (no activation function, as we'll apply softmax later)
        
        return x

In [ ]:
# Randomize with manual seed
torch.manual_seed(41)

# Create an instance of the model
model = Model(in_features=4, h1 = 8, h2 = 9, out_features=3)

# Load Data

In [ ]:
data = uciml.fetch_ucirepo("iris").data
df = data.original
df.head()

## Converting X and y in numpy array of float

In [ ]:
X = df.drop("class", axis=1).values
y = df["class"].replace("Iris-setosa", 0.0).replace("Iris-versicolor", 1.0).replace("Iris-virginica", 2.0).astype(np.float64).values

# Modellazation

## Splitting in train and test set

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=41)

## Converting the data in tensors

In [ ]:
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor = torch.FloatTensor(X_test)
y_train_tensor = torch.LongTensor(y_train)
y_test_tensor = torch.LongTensor(y_test)

## Set criterion and optimezer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
model.parameters

## Training Model

In [ ]:
# set epochs
epochs = 100
losses = []

for i in range(epochs):
    # Forward propagation
    y_pred = model.forward(X_train_tensor)
    
    loss = criterion(y_pred, y_train_tensor) # calculate the loss
    losses.append(loss.detach().numpy()) # keep track of the loss

    # print every 10 epochs
    if i % 10 == 0:
        print(f"Epoch: {i} Loss: {loss}")
    
    # Backward propagation
    optimizer.zero_grad() # zero the gradients
    loss.backward() # backpropagate the loss
    optimizer.step() # update the weights

In [ ]:
plt.plot(range(epochs), losses)
plt.ylabel("Loss")
plt.xlabel("Epochs")